In [3]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 MB 38.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.5-cp312-cp312-macosx_14_0_arm64.whl size=3090784 sha256=847afe6b0e974ea151550773c8d86c8c86433b91e7d37c53f7f424c85935e36d
  Stored in directory: /Users/vishak/Library/Caches/pip/wheels/89/78/d1/18413058f6864ae554a299e30b0ab159383ba57a5fb1cd4c54
Successfully built llama-cpp-python


In [15]:
# Create a models directory
!mkdir -p models

# Download the quantized model into the 'models' directory using curl
!curl -L -o models/mistral-7b-instruct-v0.1.Q4_K_M.gguf https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1139  100  1139    0     0  11161      0 --:--:-- --:--:-- --:--:-- 11277
100 4166M  100 4166M    0     0  43.2M      0  0:01:36  0:01:36 --:--:-- 41.7M0  40.5M      0  0:01:42  0:00:03  0:01:39 43.7MM 0  43.2M      0  0:01:36  0:01:13  0:00:23 42.7M


In [37]:
import os
from typing import Dict, List, Optional, Tuple
from llama_cpp import Llama

class InteractiveCodeTools:
    def __init__(self, model_path: str = "mistral-7b-instruct-v0.1.Q4_K_M.gguf"):
        """Initialize the CodeTools with local Mistral model."""
        if not os.path.exists(model_path):
            self.download_model()
        
        self.llm = Llama(
            model_path=model_path,
            n_ctx=8192,
            n_threads=4,
            n_gpu_layers=0
        )
        
        self.supported_languages = {
            "1": "Python",
            "2": "JavaScript",
            "3": "Java",
            "4": "HTML/CSS",
            "5": "React",
            "6": "Node.js"
        }

    def download_model(self):
        """Download the Mistral model if not present."""
        import wget
        import os

        model_url = "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
        model_dir = "models"
        
        if not os.path.exists(model_dir):
            os.makedirs(model_dir)
        
        print("Downloading Mistral model... This might take a while...")
        wget.download(model_url, os.path.join(model_dir, "mistral-7b-instruct-v0.1.Q4_K_M.gguf"))
        print("\nModel downloaded successfully!")

    def get_main_menu_choice(self) -> str:
        """Get the main operation choice from user."""
        print("\nWhat would you like to do?")
        print("1. Generate new code from prompt")
        print("2. Analyze existing code")
        return input("Choose an option (1/2): ")

    def get_language_choice(self) -> str:
        """Get the programming language choice from user."""
        print("\nSupported Languages:")
        for key, language in self.supported_languages.items():
            print(f"{key}. {language}")
        choice = input("Choose a language (1-6): ")
        return self.supported_languages.get(choice, "Python")

    def get_code_generation_input(self) -> Tuple[str, str]:
        """Get requirements for code generation."""
        language = self.get_language_choice()
        print("\nEnter your requirements/prompt for code generation:")
        print("(Describe what you want the code to do, any specific features or constraints)")
        prompt = input("\nPrompt: ")
        return language, prompt

    def get_code_analysis_input(self) -> Tuple[str, str, str]:
        """Get code and analysis preferences from user."""
        language = self.get_language_choice()
        
        print("\nCode Input Options:")
        print("1. Enter code directly")
        print("2. Load from file")
        choice = input("Choose an option (1/2): ")

        if choice == "1":
            print("\nEnter your code (type 'END' on a new line when finished):")
            code_lines = []
            while True:
                line = input()
                if line.strip() == 'END':
                    break
                code_lines.append(line)
            code = "\n".join(code_lines)
        else:
            file_path = input("\nEnter the path to your code file: ")
            with open(file_path, 'r') as f:
                code = f.read()

        print("\nAnalysis Types:")
        print("1. All (optimization, security/accessibility, style, and correction)")
        print("2. Optimization only")
        print("3. Security/Accessibility only")
        print("4. Style only")
        print("5. Code correction only")
        
        analysis_choice = input("Choose analysis type (1/2/3/4/5): ")
        analysis_map = {
            "1": "all",
            "2": "optimization",
            "3": "security",
            "4": "style",
            "5": "correction"
        }
        
        return language, code, analysis_map.get(analysis_choice, "all")

    def generate_code(self, language: str, prompt: str) -> str:
        """Generate code based on user requirements."""
        generation_prompt = f"""<s>[INST] As an expert {language} developer, please generate code based on the following requirements:

{prompt}

Please provide:
1. Complete, working code that fulfills the requirements
2. Brief explanation of how the code works
3. Any assumptions made
4. Usage examples

Make sure the code follows best practices, includes proper error handling, and is well-documented. [/INST]"""

        try:
            result = self.query_model(generation_prompt, max_tokens=2000)
            return result
        except Exception as e:
            return f"Error during code generation: {str(e)}"

    def analyze_code(self, language: str, code: str, analysis_type: str = "all") -> Dict:
        """Analyze code based on user preferences."""
        prompts = {
            "optimization": f"""You are an expert {language} developer. Analyze this code for performance optimization opportunities. Consider:
                1. Time complexity
                2. Space complexity
                3. Algorithm efficiency
                4. Resource usage
                Provide specific suggestions with examples.""",
            
            "security": f"""You are an expert security engineer. Analyze this {language} code for security vulnerabilities. Consider:
                1. Input validation
                2. Data sanitization
                3. Common security pitfalls
                4. Best security practices
                Provide specific fixes with examples.""",
            
            "style": f"""You are an expert {language} developer. Review this code for style improvements. Consider:
                1. Language-specific conventions
                2. Code organization
                3. Documentation
                4. Naming conventions
                Provide specific suggestions with examples.""",
                
            "correction": f"""You are an expert {language} developer. Review this code for errors and potential bugs. Consider:
                1. Syntax errors
                2. Logical errors
                3. Edge cases
                4. Exception handling
                Provide the corrected code with explanations of the changes made."""
        }

        results = {}
        analysis_types = ["optimization", "security", "style", "correction"] if analysis_type == "all" else [analysis_type]

        for current_type in analysis_types:
            prompt = f"""<s>[INST] As an expert code reviewer, please analyze this {language} code:

{code}

{prompts[current_type]}

Format your response as:
1. Summary of findings
2. Detailed suggestions (with code examples)
3. Priority level for each suggestion (High/Medium/Low)
{'4. Complete corrected code' if current_type == 'correction' else ''} [/INST]"""

            try:
                results[current_type] = self.query_model(prompt)
            except Exception as e:
                results[current_type] = f"Error during {current_type} analysis: {str(e)}"

        return results

    def query_model(self, prompt: str, max_tokens: int = 1500) -> str:
        """Query the local Mistral model."""
        try:
            response = self.llm(
                prompt,
                max_tokens=max_tokens,
                temperature=0.7,
                stop=["</s>", "[/INST]"]
            )
            return response['choices'][0]['text'].strip()
        except Exception as e:
            return f"Error querying model: {str(e)}"

    def format_results(self, results: Dict, is_generation: bool = False) -> str:
        """Format analysis or generation results into a readable report."""
        if is_generation:
            formatted_output = "\nCODE GENERATION RESULTS\n" + "=" * 50 + "\n\n"
            formatted_output += results
        else:
            formatted_output = "\nCODE ANALYSIS REPORT\n" + "=" * 50 + "\n\n"
            for analysis_type, result in results.items():
                formatted_output += f"\n{analysis_type.upper()} ANALYSIS\n{'-' * 30}\n"
                formatted_output += f"{result}\n\n"
        
        return formatted_output

def main():
    """Interactive main function."""
    print("Welcome to the Code Generator and Analyzer!")
    
    try:
        tool = InteractiveCodeTools()
        while True:
            choice = tool.get_main_menu_choice()
            
            if choice == "1":
                # Generate code
                language, prompt = tool.get_code_generation_input()
                results = tool.generate_code(language, prompt)
                print(tool.format_results(results, is_generation=True))
            else:
                # Analyze code
                language, code, analysis_type = tool.get_code_analysis_input()
                results = tool.analyze_code(language, code, analysis_type)
                print(tool.format_results(results))
            
            if input("\nWould you like to do something else? (y/n): ").lower() != 'y':
                break
                
    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    main()

llama_load_model_from_file: using device Metal (Apple M3 Pro) - 12261 MiB free
llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count 

Welcome to the Code Generator and Analyzer!


ggml_metal_init: loaded kernel_mul_mv_ext_q4_K_f32_r1_5               0x1218f95f0 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q5_K_f32_r1_2               0x1047902c0 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q5_K_f32_r1_3               0x1068dd5c0 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q5_K_f32_r1_4               0x1068dd7f0 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q5_K_f32_r1_5               0x168205a50 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q6_K_f32_r1_2               0x1218f9820 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q6_K_f32_r1_3               0x168205c80 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q6_K_f32_r1_4               0x1068dda20 | th_max = 1024 | th_width =   32
ggml_metal_init: loaded kernel_mul_mv_ext_q6_K_f32_r1_5               0x


What would you like to do?
1. Generate new code from prompt
2. Analyze existing code


KeyboardInterrupt: Interrupted by user